# Week 4 - DESI pseudo-labels

Run from the cloned repository. Use native DESI flux arrays and the pinned split.

In [ ]:
%pip install 'numpy<2' 'scipy<1.16' 'astropy<7' 'photutils==2.3.0' 'pandas<3' 'pyarrow<22' huggingface_hub

In [ ]:
# Importing necessary libraries
import json
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Week-4").exists())
W4, W5 = REPO / "Week-4", REPO / "Week-5"
sys.path[:0] = [str(W4 / "src"), str(W5 / "src")]


In [ ]:
# Loading the Week 1 survey cutouts
from huggingface_hub import hf_hub_download
from cutouts import prepare_cutouts, pseudo_label

DATASET = "MultimodalUniverse/legacysurvey"
REVISION = "112bddb46da031de2807ced8fe28fb79ed13b515"
files = [Path(hf_hub_download(
    DATASET, f"dr10_south_21/train-{i:05d}-of-00165.parquet",
    repo_type="dataset", revision=REVISION,
)) for i in range(6)]
splits = prepare_cutouts(W4, files, REVISION)
print({name: len(splits[name]) for name in ("train", "validation", "test")})


In [ ]:
# Inspecting the image, segments, and three-class mask
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["black", "gold", "royalblue"])
available = [key for key in splits["train"] if (W4 / f"data/cutouts/{key}.npz").exists()]
ids = np.random.default_rng(42).choice(available, 50, replace=False)
output = W4 / "results/pseudo_labels"
for page in range(5):
    fig, axes = plt.subplots(10, 3, figsize=(9, 24))
    for row, key in enumerate(ids[page * 10:(page + 1) * 10]):
        with np.load(W4 / f"data/cutouts/{key}.npz") as sample:
            instances, mask, target = pseudo_label(sample["griz"][[0, 1, 3]])
            panels = (sample["rgb"], instances, mask)
        for ax, panel, title in zip(axes[row], panels, (key, "Segments", "3-class mask")):
            ax.imshow(panel, cmap=cmap if title == "3-class mask" else None,
                      vmin=0 if title == "3-class mask" else None,
                      vmax=2 if title == "3-class mask" else None)
            ax.set_title(title)
            ax.axis("off")
    fig.tight_layout()
    fig.savefig(output / f"review_{page + 1}.png", dpi=110)
    plt.show()


## Reviewing masks

Look for fragments, faint outskirts, and merged neighbours. Record clearly broken IDs in `data/excluded_ids.json` before training. For the released split, keep `splits.json` fixed. The hard set is a diagnostic subset of test and is never used for fitting or validation.